# Employee DWA Report Generation
## Multi-Section Analysis Pipeline
---

This notebook processes Apple employee data through three analysis sections:
- **Section 0**: Creates master parquet with DWA mapping by employee
- **Section 1**: Creates ONET-clustered aggregated data
- **Section 2**: TBD

In [26]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime
import os

print("Libraries loaded successfully")
print(f"Processing started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Libraries loaded successfully
Processing started at: 2026-01-18 16:26:49


---
# SECTION 0: Master Parquet Creation
### Employee-level DWA expansion with weighted salaries
---

In [27]:
# Load the Apple employee data and remove duplicates
print("Loading Apple employee data...")
company_file = 'data/AAPL.parquet'
company_code = company_file.split('/')[-1].replace('.parquet', '')
df_employees = pd.read_parquet(company_file)

print(f"✓ Company code: {company_code}")
print(f"✓ Loaded {len(df_employees):,} records")
print(f"✓ Unique user_ids: {df_employees['user_id'].nunique():,}")

# Drop the 'rn' column if it exists (row number from previous processing)
# if 'rn' in df_employees.columns:
#     df_employees = df_employees.drop(columns=['rn'])
#     print(f"✓ Removed 'rn' column")

# Remove duplicate user_ids, keeping the first occurrence
df_employees = df_employees.drop_duplicates(subset='user_id', keep='first')

print(f"\nAfter removing duplicates:")
print(f"✓ Deduplicated records: {len(df_employees):,}")
print(f"✓ Unique user_ids: {df_employees['user_id'].nunique():,}")
print(f"✓ Unique onet_codes: {df_employees['onet_code'].nunique():,}")
print(f"\nColumns in employee data: {df_employees.columns.tolist()}")

Loading Apple employee data...
✓ Company code: AAPL
✓ Loaded 287,161 records
✓ Unique user_ids: 287,161

After removing duplicates:
✓ Deduplicated records: 287,161
✓ Unique user_ids: 287,161
✓ Unique onet_codes: 803

Columns in employee data: ['user_id', 'position_id', 'region', 'company_raw', 'country', 'state', 'location_raw', 'metro_area', 'msa', 'city', 'remote_suitability', 'weight', 'end_salary', 'title_raw', 'seniority', 'salary', 'description', 'onet_title', 'ultimate_parent_rcid', 'onet_code', 'prestige', 'highest_degree', 'sex_predicted', 'ethnicity_predicted', 'rn']


In [28]:
# Display sample of employee data
print("\n=== Sample Employee Data ===")
display(df_employees.head())
print(f"\nData shape: {df_employees.shape}")


=== Sample Employee Data ===


,user_id,position_id,region,company_raw,country,state,location_raw,metro_area,msa,city,...,salary,description,onet_title,ultimate_parent_rcid,onet_code,prestige,highest_degree,sex_predicted,ethnicity_predicted,rn
1,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,68894.41,<NA>,Information Technology Project Managers,1232095.0,15-1299.09,0.587231,Master,F,White,1
4,1007301.0,2292766297204852736.0,Northern America,Apple,United States,California,"Sunnyvale, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Sunnyvale,...,222234.24,<NA>,Penetration Testers,1232095.0,15-1299.04,0.632171,Bachelor,M,API,1
5,1012376.0,-2039848491300531968.0,Northern America,Apple,United States,Texas,"Austin, Texas, United States",austin metropolitan area,Austin-Round Rock TX MSA,Austin,...,63091.21,<NA>,Billing and Posting Clerks,1232095.0,43-3021.00,1.249455,<NA>,F,White,1
6,1013951.0,3001738349142168576.0,Northern America,Apple,United States,California,"Cupertino, CA",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,107023.99,<NA>,Photonics Engineers,1232095.0,17-2199.07,1.102509,Doctor,F,API,1
8,1016615.0,6400852030022673408.0,Northern America,Apple,United States,Texas,"Austin, Texas, United States",austin metropolitan area,Austin-Round Rock TX MSA,Austin,...,104722.76,<NA>,"Sales Representatives of Services, Except Adve...",1232095.0,41-3091.00,0.852469,Bachelor,M,Hispanic,1



Data shape: (287161, 25)


In [29]:
# Load the DWA (Detailed Work Activities) mapping data
print("\nLoading DWA time normalized data...")
df_dwa = pd.read_parquet('data/dwa_times_normalized.parquet')

print(f"✓ Loaded {len(df_dwa):,} DWA records")
print(f"✓ Unique onet_codes in DWA data: {df_dwa['O*NET-SOC Code'].nunique():,}")
print(f"\nColumns in DWA data: {df_dwa.columns.tolist()}")


Loading DWA time normalized data...
✓ Loaded 18,358 DWA records
✓ Unique onet_codes in DWA data: 923

Columns in DWA data: ['O*NET-SOC Code', 'Title', 'DWA Title', 'DWA ID', 'dwa_time_normalized', 'ARS']


In [30]:
# Display sample of DWA data
print("\n=== Sample DWA Data ===")
display(df_dwa.head())
print(f"\nDWA data shape: {df_dwa.shape}")


=== Sample DWA Data ===


,O*NET-SOC Code,Title,DWA Title,DWA ID,dwa_time_normalized,ARS
0,11-1011.00,Chief Executives,Advise others on legal or regulatory complianc...,4.A.4.b.6.I08.D04,0.001892,0.634327
1,11-1011.00,Chief Executives,Analyze data to assess operational or project ...,4.A.2.a.4.I07.D09,0.016559,0.383694
2,11-1011.00,Chief Executives,Analyze data to inform operational decisions o...,4.A.2.a.4.I07.D12,0.026878,0.314204
3,11-1011.00,Chief Executives,Analyze impact of legal or regulatory changes.,4.A.2.a.4.I09.D03,0.001892,0.598174
4,11-1011.00,Chief Executives,Communicate organizational policies and proced...,4.A.4.a.1.I02.D03,0.020659,0.500648



DWA data shape: (18358, 6)


In [31]:
# Check the structure and identify the correct column names
print("\n=== Checking DWA Data Structure ===")
print("\nDWA columns:")
for col in df_dwa.columns:
    print(f"  - {col}: {df_dwa[col].dtype}")

print("\n=== Sample DWA entries for one onet_code ===")
sample_onet = df_dwa['O*NET-SOC Code'].iloc[0]
print(f"\nShowing DWAs for onet_code: {sample_onet}")
display(df_dwa[df_dwa['O*NET-SOC Code'] == sample_onet].head(10))


=== Checking DWA Data Structure ===

DWA columns:
  - O*NET-SOC Code: object
  - Title: object
  - DWA Title: object
  - DWA ID: object
  - dwa_time_normalized: float64
  - ARS: float64

=== Sample DWA entries for one onet_code ===

Showing DWAs for onet_code: 11-1011.00


,O*NET-SOC Code,Title,DWA Title,DWA ID,dwa_time_normalized,ARS
0,11-1011.00,Chief Executives,Advise others on legal or regulatory complianc...,4.A.4.b.6.I08.D04,0.001892,0.634327
1,11-1011.00,Chief Executives,Analyze data to assess operational or project ...,4.A.2.a.4.I07.D09,0.016559,0.383694
2,11-1011.00,Chief Executives,Analyze data to inform operational decisions o...,4.A.2.a.4.I07.D12,0.026878,0.314204
3,11-1011.00,Chief Executives,Analyze impact of legal or regulatory changes.,4.A.2.a.4.I09.D03,0.001892,0.598174
4,11-1011.00,Chief Executives,Communicate organizational policies and proced...,4.A.4.a.1.I02.D03,0.020659,0.500648
5,11-1011.00,Chief Executives,Conduct hearings to investigate legal issues.,4.A.1.a.1.I03.D06,0.050855,0.656230
6,11-1011.00,Chief Executives,Conduct research on social issues.,4.A.1.a.1.I18.D03,0.003472,0.425920
7,11-1011.00,Chief Executives,Conduct research to gain information about pro...,4.A.1.a.1.I20.D04,0.003472,0.409384
8,11-1011.00,Chief Executives,Confer with organizational members to accompli...,4.A.4.a.2.I03.D14,0.014783,0.455492
9,11-1011.00,Chief Executives,Coordinate special events or programs.,4.A.4.b.4.I11.D03,0.041854,0.476621


---
## Perform Left Join: Employee Data + DWA Information

Each employee will be expanded to multiple rows based on their O*NET code's associated DWAs.

In [32]:
# Perform the left join on onet_code
print("\n" + "="*70)
print("PERFORMING LEFT JOIN: EMPLOYEES + DWA DATA")
print("="*70)

print(f"\nBefore join:")
print(f"  Employee records: {len(df_employees):,}")
print(f"  DWA records: {len(df_dwa):,}")

# Perform left join on onet_code
df_result = df_employees.merge(
    df_dwa,
    left_on='onet_code',
    right_on='O*NET-SOC Code',
    suffixes=('', '_dwa')
)

print(f"\nAfter join:")
print(f"  Total records: {len(df_result):,}")
print(f"  Unique user_ids: {df_result['user_id'].nunique():,}")
print(f"  Average DWAs per employee: {len(df_result) / df_employees['user_id'].nunique():.2f}")

print(f"\n✓ Join completed successfully!")


PERFORMING LEFT JOIN: EMPLOYEES + DWA DATA

Before join:
  Employee records: 287,161
  DWA records: 18,358

After join:
  Total records: 5,701,836
  Unique user_ids: 267,174
  Average DWAs per employee: 19.86

✓ Join completed successfully!


In [33]:
# Display sample of the merged data
print("\n=== Sample of Merged Data ===")
print(f"\nShowing all DWAs for first user_id:")
sample_user = df_result['user_id'].iloc[0]
print(f"User ID: {sample_user}")
display(df_result[df_result['user_id'] == sample_user].head(20))

print(f"\n=== Result Data Structure ===")
print(f"Total columns: {len(df_result.columns)}")
print(f"Columns: {df_result.columns.tolist()}")


=== Sample of Merged Data ===

Showing all DWAs for first user_id:
User ID: 1000569.0


,user_id,position_id,region,company_raw,country,state,location_raw,metro_area,msa,city,...,highest_degree,sex_predicted,ethnicity_predicted,rn,O*NET-SOC Code,Title,DWA Title,DWA ID,dwa_time_normalized,ARS
0,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,Analyze data to identify trends or relationshi...,4.A.2.a.4.I04.D02,0.016065,0.358460
1,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,"Analyze security of systems, network, or data.",4.A.2.a.4.I12.D02,0.020953,0.420992
2,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,Assign duties or work schedules to employees.,4.A.4.b.4.I13.D06,0.030795,0.442339
3,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,Collaborate with others to resolve information...,4.A.4.a.2.I04.D03,0.055943,0.401509
4,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,Collect data about customer needs.,4.A.1.a.1.I14.D05,0.039554,0.595644
5,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,Coordinate resource procurement activities.,4.A.4.b.4.I12.D36,0.022009,0.531268
6,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,Develop detailed project plans.,4.A.2.b.6.I02.D08,0.076568,0.582707
7,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,Develop guidelines for system implementation.,4.A.2.b.4.I03.D08,0.065742,0.566360
8,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,Develop information communication procedures.,4.A.2.b.4.I03.D03,0.024740,0.606088
9,1000569.0,4536490463696964608.0,Northern America,Apple,United States,California,"Cupertino, California",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,Cupertino,...,Master,F,White,1,15-1299.09,Information Technology Project Managers,Evaluate utility of software or hardware techn...,4.A.2.a.1.I07.D08,0.031570,0.382371



=== Result Data Structure ===
Total columns: 31
Columns: ['user_id', 'position_id', 'region', 'company_raw', 'country', 'state', 'location_raw', 'metro_area', 'msa', 'city', 'remote_suitability', 'weight', 'end_salary', 'title_raw', 'seniority', 'salary', 'description', 'onet_title', 'ultimate_parent_rcid', 'onet_code', 'prestige', 'highest_degree', 'sex_predicted', 'ethnicity_predicted', 'rn', 'O*NET-SOC Code', 'Title', 'DWA Title', 'DWA ID', 'dwa_time_normalized', 'ARS']


In [34]:
# Check for any null values in key DWA columns
print("\n=== Data Quality Check ===")
print("\nNull values in DWA-related columns:")
dwa_cols = [col for col in df_result.columns if 'dwa' in col.lower() or col in ['ARS']]
for col in dwa_cols:
    if col in df_result.columns:
        null_count = df_result[col].isnull().sum()
        null_pct = (null_count / len(df_result) * 100)
        print(f"  {col}: {null_count:,} ({null_pct:.2f}%)")

# Check employees with no DWA matches
employees_no_dwa = df_result[df_result[dwa_cols].isnull().all(axis=1)]['user_id'].nunique()
print(f"\nEmployees with no DWA matches: {employees_no_dwa:,} ({employees_no_dwa/df_employees['user_id'].nunique()*100:.2f}%)")


=== Data Quality Check ===

Null values in DWA-related columns:
  DWA Title: 0 (0.00%)
  DWA ID: 0 (0.00%)
  dwa_time_normalized: 262,256 (4.60%)
  ARS: 541 (0.01%)

Employees with no DWA matches: 0 (0.00%)


---
## Summary Statistics

---
## Data Transformations & Enrichment

Calculate DWA-weighted salary and add occupation-level exposure data.

In [35]:
# Calculate DWA-weighted salary using 'salary' column
print("\n" + "="*70)
print("CALCULATING DWA-WEIGHTED SALARY")
print("="*70)

# Check if salary and dwa_time_normalized columns exist
print(f"\nChecking for required columns...")
print(f"  'salary' exists: {'salary' in df_result.columns}")
print(f"  'dwa_time_normalized' exists: {'dwa_time_normalized' in df_result.columns}")

# Drop end_salary if it exists
if 'end_salary' in df_result.columns:
    df_result = df_result.drop(columns=['end_salary'])
    print(f"  ✓ Dropped 'end_salary' column - using 'salary' instead")

# Create the weighted salary column
df_result['dwa_weighted_salary'] = df_result['salary'] * df_result['dwa_time_normalized']

print(f"\n✓ Created 'dwa_weighted_salary' column")
print(f"  Sample values:")
print(f"    salary: {df_result['salary'].iloc[0]:,.2f}")
print(f"    dwa_time_normalized: {df_result['dwa_time_normalized'].iloc[0]:.4f}")
print(f"    dwa_weighted_salary: {df_result['dwa_weighted_salary'].iloc[0]:,.2f}")

# Display summary statistics
print(f"\n📊 DWA-Weighted Salary Statistics:")
print(f"  Mean: ${df_result['dwa_weighted_salary'].mean():,.2f}")
print(f"  Median: ${df_result['dwa_weighted_salary'].median():,.2f}")
print(f"  Min: ${df_result['dwa_weighted_salary'].min():,.2f}")
print(f"  Max: ${df_result['dwa_weighted_salary'].max():,.2f}")


CALCULATING DWA-WEIGHTED SALARY

Checking for required columns...
  'salary' exists: True
  'dwa_time_normalized' exists: True


  ✓ Dropped 'end_salary' column - using 'salary' instead

✓ Created 'dwa_weighted_salary' column
  Sample values:
    salary: 68,894.41
    dwa_time_normalized: 0.0161
    dwa_weighted_salary: 1,106.76

📊 DWA-Weighted Salary Statistics:
  Mean: $5,112.25
  Median: $2,669.24
  Min: $0.00
  Max: $191,816.97


In [36]:
# Load occupation-level exposure data from pierre_occ_level.csv (for reference/validation only)
print("\n" + "="*70)
print("NOTE: DWA-LEVEL EXPOSURE ALREADY IN DATA")
print("="*70)

print(f"\n✓ DWA-level exposure (ARS) already loaded from dwa_times_normalized.parquet")
print(f"  Each DWA has a unique exposure value")

# Rename ARS to exposure for consistency with other notebooks
if 'ARS' in df_result.columns:
    df_result = df_result.rename(columns={'ARS': 'exposure'})
    print(f"  ✓ Renamed 'ARS' column to 'exposure'")

# Check exposure data quality
print(f"\n📊 Exposure Data Statistics:")
print(f"  Records with exposure: {(~df_result['exposure'].isnull()).sum():,}")
print(f"  Exposure range: {df_result['exposure'].min():.3f} to {df_result['exposure'].max():.3f}")
print(f"  Unique DWA exposure values: {df_result['exposure'].nunique():,}")
print(f"  Unique DWAs (DWA ID): {df_result['DWA ID'].nunique():,}")

# Verify each DWA has unique exposure
exposure_per_dwa = df_result.groupby('DWA ID')['exposure'].nunique()
if (exposure_per_dwa > 1).any():
    print(f"  ⚠️ WARNING: Some DWAs have multiple exposure values!")
else:
    print(f"  ✓ Each DWA has exactly one exposure value (correct!)")

print(f"\n✓ Using DWA-level exposure (not occupation-level)!")


NOTE: DWA-LEVEL EXPOSURE ALREADY IN DATA

✓ DWA-level exposure (ARS) already loaded from dwa_times_normalized.parquet
  Each DWA has a unique exposure value
  ✓ Renamed 'ARS' column to 'exposure'

📊 Exposure Data Statistics:
  Records with exposure: 5,701,295
  Exposure range: 0.060 to 0.996
  Unique DWA exposure values: 2,050
  Unique DWAs (DWA ID): 2,054
  ✓ Each DWA has exactly one exposure value (correct!)

✓ Using DWA-level exposure (not occupation-level)!


In [37]:
# Generate summary statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

print(f"\n📊 Overall Statistics:")
print(f"  Original employees: {df_employees['user_id'].nunique():,}")
print(f"  Expanded records: {len(df_result):,}")
print(f"  Expansion ratio: {len(df_result) / len(df_employees):.2f}x")

# DWAs per employee distribution
dwas_per_employee = df_result.groupby('user_id').size()
print(f"\n📊 DWAs per Employee:")
print(f"  Mean: {dwas_per_employee.mean():.2f}")
print(f"  Median: {dwas_per_employee.median():.0f}")
print(f"  Min: {dwas_per_employee.min()}")
print(f"  Max: {dwas_per_employee.max()}")
print(f"  Std Dev: {dwas_per_employee.std():.2f}")

# Top positions by DWA count
print(f"\n📊 Top 5 O*NET Codes by DWA Count:")
dwas_per_onet = df_result.groupby('onet_code').size().sort_values(ascending=False).head(5)
for onet, count in dwas_per_onet.items():
    onet_title = df_result[df_result['onet_code'] == onet]['onet_title'].iloc[0] if 'onet_title' in df_result.columns else 'N/A'
    print(f"  {onet} ({onet_title}): {count:,} DWAs")


SUMMARY STATISTICS

📊 Overall Statistics:
  Original employees: 287,161
  Expanded records: 5,701,836
  Expansion ratio: 19.86x

📊 DWAs per Employee:
  Mean: 21.34
  Median: 22
  Min: 5
  Max: 39
  Std Dev: 5.20

📊 Top 5 O*NET Codes by DWA Count:
  15-1232.00 (Computer User Support Specialists): 601,216 DWAs
  15-1252.00 (Software Developers): 465,138 DWAs
  41-2031.00 (Retail Salespersons): 316,584 DWAs
  41-4012.00 (Sales Representatives, Wholesale and Manufacturing, Except Technical and Scientific Products): 279,084 DWAs
  15-1299.08 (Computer Systems Engineers/Architects): 254,610 DWAs


---
## Export Results

In [38]:
# Create output directory if it doesn't exist
output_dir = 'output/Tables'
os.makedirs(output_dir, exist_ok=True)

# Generate output filename based on company code (will overwrite)
output_file = f'{output_dir}/{company_code}_employees_with_DWAs.parquet'

# Delete existing file if it exists to ensure clean overwrite
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"\n🗑️  Removed existing file: {output_file}")

print(f"\n💾 Saving results to: {output_file}")
print(f"   Records to save: {len(df_result):,}")
print(f"   Columns: {len(df_result.columns)}")
print(f"   Key columns included: dwa_weighted_salary, exposure")

# Save to parquet
df_result.to_parquet(output_file, index=False, compression='snappy')


# Verify file was created and get sizeprint(f"   Location: {os.path.abspath(output_file)}")

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"   File size: {file_size_mb:.2f} MB")
print(f"\n✓ File saved successfully!")


🗑️  Removed existing file: output/Tables/AAPL_employees_with_DWAs.parquet

💾 Saving results to: output/Tables/AAPL_employees_with_DWAs.parquet
   Records to save: 5,701,836
   Columns: 31
   Key columns included: dwa_weighted_salary, exposure
   File size: 165.73 MB

✓ File saved successfully!


In [39]:
# Verify the saved file by reading it back
print("\n🔍 Verifying saved file...")
df_verify = pd.read_parquet(output_file)

print(f"✓ Verification successful!")
print(f"  Records read: {len(df_verify):,}")
print(f"  Columns: {len(df_verify.columns)}")
print(f"  Match: {len(df_verify) == len(df_result)}")

print(f"\n" + "="*70)
print("PROCESSING COMPLETED SUCCESSFULLY")
print("="*70)
print(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


🔍 Verifying saved file...
✓ Verification successful!
  Records read: 5,701,836
  Columns: 31
  Match: True

PROCESSING COMPLETED SUCCESSFULLY
Finished at: 2026-01-18 16:27:22


---
## Sample Output Preview

In [40]:
# Display a comprehensive sample of one employee's expanded data
print("\n=== FINAL OUTPUT SAMPLE ===")
print("\nShowing complete DWA expansion for a sample employee:\n")

# Pick an employee with a reasonable number of DWAs
dwas_per_user = df_result.groupby('user_id').size()
median_dwa_count = dwas_per_user.median()
sample_user = dwas_per_user[dwas_per_user == median_dwa_count].index[0] if len(dwas_per_user[dwas_per_user == median_dwa_count]) > 0 else dwas_per_user.index[0]

sample_output = df_result[df_result['user_id'] == sample_user]
print(f"User ID: {sample_user}")
print(f"Number of DWAs: {len(sample_output)}")
if 'onet_title' in sample_output.columns:
    print(f"Position: {sample_output['onet_title'].iloc[0]}")
if 'onet_code' in sample_output.columns:
    print(f"O*NET Code: {sample_output['onet_code'].iloc[0]}")

print("\n")
display(sample_output)


=== FINAL OUTPUT SAMPLE ===

Showing complete DWA expansion for a sample employee:

User ID: 1055966.0
Number of DWAs: 22
Position: Computer User Support Specialists
O*NET Code: 15-1232.00




,user_id,position_id,region,company_raw,country,state,location_raw,metro_area,msa,city,...,sex_predicted,ethnicity_predicted,rn,O*NET-SOC Code,Title,DWA Title,DWA ID,dwa_time_normalized,exposure,dwa_weighted_salary
315,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Collaborate with others to determine design sp...,4.A.4.a.2.I09.D07,0.049531,0.458391,5719.723083
316,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Collaborate with others to resolve information...,4.A.4.a.2.I04.D03,0.060629,0.401509,7001.237224
317,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Conduct research to gain information about pro...,4.A.1.a.1.I20.D04,0.003093,0.409384,357.153108
318,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Document operational activities.,4.A.3.b.6.I08.D11,0.104368,0.506482,12052.182126
319,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Evaluate utility of software or hardware techn...,4.A.2.a.1.I07.D08,0.005040,0.382371,582.033422
320,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Install computer hardware.,4.A.3.b.1.I03.D02,0.088547,0.299028,10225.169896
321,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Install computer software.,4.A.3.b.1.I03.D01,0.020706,0.272239,2391.046495
322,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Maintain computer hardware.,4.A.3.b.4.I04.D01,0.020706,0.334560,2391.046495
323,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Modify software programs to improve performance.,4.A.2.b.2.I02.D07,0.007651,0.334462,883.543551
324,1055966.0,-2982276359854843904.0,Northern America,Apple,United States,California,"San Jose, California, United States",san jose metropolitan area (california),San Jose-Sunnyvale-Santa Clara CA MSA,San Jose,...,F,Hispanic,1,15-1232.00,Computer User Support Specialists,Monitor computer system performance to ensure ...,4.A.1.a.2.I10.D03,0.115107,0.303318,13292.290759


---
# SECTION 1: ONET-Level Clustering
### Aggregate data by O*NET code with demographic and salary information
---

In [41]:
# Create ONET-level aggregated dataset
print("\n" + "="*70)
print("SECTION 1: CREATING ONET-LEVEL AGGREGATED DATA")
print("="*70)

print(f"\nStarting aggregation by O*NET code...")
print(f"  Input records: {len(df_result):,}")
print(f"  Unique O*NET codes: {df_result['onet_code'].nunique():,}")

# Group by onet_code and aggregate
# Note: We need to deduplicate user_id before summing salary to avoid double-counting
df_onet_temp = df_result.groupby(['onet_code', 'user_id']).agg({
    'onet_title': 'first',
    'remote_suitability': 'first',
    'salary': 'first',  # Each user_id has the same salary across all DWA rows
    'seniority': 'first',
    'sex_predicted': 'first',
    'ethnicity_predicted': 'first'
}).reset_index()

# Now aggregate by onet_code with deduplicated user data
df_onet_clustered = df_onet_temp.groupby('onet_code').agg({
    # Keep first occurrence of descriptive fields
    'onet_title': 'first',
    'remote_suitability': 'first',
    
    # Count number of unique employees per occupation
    'user_id': 'count',  # Now each row is unique per user
    
    # Sum salaries to get total wage bill (now deduplicated)
    'salary': 'sum',
    
    # Most common seniority (mode)
    'seniority': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    
    # Calculate % female
    'sex_predicted': lambda x: (x == 'F').sum() / len(x) * 100,
    
    # Calculate % white
    'ethnicity_predicted': lambda x: (x == 'White').sum() / len(x) * 100
}).reset_index()

# Count unique DWAs per occupation from original df_result
dwa_counts = df_result.groupby('onet_code')['DWA ID'].nunique().reset_index()
dwa_counts.columns = ['onet_code', 'n_dwa_per_occupation']

# Merge DWA counts back
df_onet_clustered = df_onet_clustered.merge(dwa_counts, on='onet_code', how='left')

# Rename columns for clarity
df_onet_clustered = df_onet_clustered.rename(columns={
    'user_id': 'n_employees',
    'salary': 'total_wagebill',
    'seniority': 'most_common_seniority',
    'sex_predicted': 'pct_female',
    'ethnicity_predicted': 'pct_white',
    'DWA ID': 'n_dwa_per_occupation'
})

print(f"\n✓ Aggregation complete!")
print(f"  Output records: {len(df_onet_clustered):,}")
print(f"  Columns: {df_onet_clustered.columns.tolist()}")

print(f"\n📊 Aggregated Data Summary:")
print(f"  Total employees: {df_onet_clustered['n_employees'].sum():,}")
print(f"  Average employees per ONET: {df_onet_clustered['n_employees'].mean():.2f}")
print(f"  Total wage bill: ${df_onet_clustered['total_wagebill'].sum():,.2f}")
print(f"  Average wage bill per ONET: ${df_onet_clustered['total_wagebill'].mean():,.2f}")
print(f"  Average % female: {df_onet_clustered['pct_female'].mean():.2f}%")
print(f"  Average % white: {df_onet_clustered['pct_white'].mean():.2f}%")
print(f"  Average DWAs per occupation: {df_onet_clustered['n_dwa_per_occupation'].mean():.2f}")


SECTION 1: CREATING ONET-LEVEL AGGREGATED DATA

Starting aggregation by O*NET code...
  Input records: 5,701,836
  Unique O*NET codes: 720

✓ Aggregation complete!
  Output records: 720
  Columns: ['onet_code', 'onet_title', 'remote_suitability', 'n_employees', 'total_wagebill', 'most_common_seniority', 'pct_female', 'pct_white', 'n_dwa_per_occupation']

📊 Aggregated Data Summary:
  Total employees: 267,174
  Average employees per ONET: 371.07
  Total wage bill: $29,648,555,247.69
  Average wage bill per ONET: $41,178,548.96
  Average % female: 35.30%
  Average % white: 60.17%
  Average DWAs per occupation: 20.07


In [42]:
# Display sample of ONET-clustered data
print("\n=== Sample ONET-Clustered Data ===")
display(df_onet_clustered.head(10))

# Show some interesting statistics
print(f"\n📊 Top 5 O*NET Codes by Total Wage Bill:")
top_wagebill = df_onet_clustered.nlargest(5, 'total_wagebill')[['onet_code', 'onet_title', 'total_wagebill', 'n_dwa_per_occupation']]
for idx, row in top_wagebill.iterrows():
    print(f"  {row['onet_code']} - {row['onet_title']}")
    print(f"    Wage bill: ${row['total_wagebill']:,.2f} | DWAs: {row['n_dwa_per_occupation']}")

print(f"\n📊 Data Quality Checks:")
print(f"  Records with remote_suitability: {(~df_onet_clustered['remote_suitability'].isnull()).sum():,}")


=== Sample ONET-Clustered Data ===


,onet_code,onet_title,remote_suitability,n_employees,total_wagebill,most_common_seniority,pct_female,pct_white,n_dwa_per_occupation
0,11-1011.00,Chief Executives,0.951461,737,166981365.48,5,22.659430,61.465400,35
1,11-1011.03,Chief Sustainability Officers,0.514857,67,9070080.09,4,52.238806,58.208955,23
2,11-1021.00,General and Operations Managers,0.5,499,58764674.78,4,38.877756,61.923848,20
3,11-1031.00,Legislators,0.495172,51,5604897.01,5,43.137255,70.588235,27
4,11-2011.00,Advertising and Promotions Managers,0.5,1353,118048141.87,2,52.475979,67.627494,33
5,11-2021.00,Marketing Managers,0.445757,5070,651475881.98,4,43.786982,62.899408,19
6,11-2022.00,Sales Managers,0.093718,2783,270978757.56,4,28.745958,71.146245,16
7,11-2032.00,Public Relations Managers,0.786355,90,9061207.97,2,50.000000,77.777778,23
8,11-2033.00,Fundraising Managers,0.5,73,9184312.88,2,46.575342,69.863014,22
9,11-3012.00,Administrative Services Managers,0.116275,1136,73400057.28,2,51.408451,64.612676,23



📊 Top 5 O*NET Codes by Total Wage Bill:
  15-1252.00 - Software Developers
    Wage bill: $4,523,945,225.16 | DWAs: 18
  15-1232.00 - Computer User Support Specialists
    Wage bill: $2,211,711,824.06 | DWAs: 22
  15-1299.08 - Computer Systems Engineers/Architects
    Wage bill: $1,185,936,618.05 | DWAs: 30
  41-2031.00 - Retail Salespersons
    Wage bill: $1,134,340,262.52 | DWAs: 24
  17-2199.06 - Microsystems Engineers
    Wage bill: $1,066,975,163.64 | DWAs: 29

📊 Data Quality Checks:
  Records with remote_suitability: 720


In [43]:
# Save ONET-clustered data
output_file_onet = f'{output_dir}/{company_code}_onet_clustered.parquet'

# Delete existing file if it exists to ensure clean overwrite
if os.path.exists(output_file_onet):
    os.remove(output_file_onet)
    print(f"\n🗑️  Removed existing file: {output_file_onet}")

print(f"\n💾 Saving ONET-clustered results to: {output_file_onet}")
df_onet_clustered.to_parquet(output_file_onet, index=False, compression='snappy')

# Verify file was created and get size
file_size_mb_onet = os.path.getsize(output_file_onet) / (1024 * 1024)
print(f"✓ File saved successfully!")
print(f"  File size: {file_size_mb_onet:.2f} MB")


🗑️  Removed existing file: output/Tables/AAPL_onet_clustered.parquet

💾 Saving ONET-clustered results to: output/Tables/AAPL_onet_clustered.parquet
✓ File saved successfully!
  File size: 0.04 MB


---
# SECTION 2: DWA-Level Clustering
### Aggregate data by DWA with employee counts and demographics
---

In [44]:
# Create DWA-level aggregated dataset
print("\n" + "="*70)
print("SECTION 2: CREATING DWA-LEVEL AGGREGATED DATA")
print("="*70)

print(f"\nStarting aggregation by DWA...")
print(f"  Input records: {len(df_result):,}")
print(f"  Unique DWAs: {df_result['DWA ID'].nunique():,}")

# Group by DWA ID and aggregate
df_dwa_clustered = df_result.groupby('DWA ID').agg({
    # Keep DWA description
    'DWA Title': 'first',
    
    # Keep first occurrence of exposure (each DWA has unique exposure)
    'exposure': 'first',
    
    # Count number of unique employees per DWA
    'user_id': 'nunique',
    
    # Total and average wage bill per DWA (using dwa_weighted_salary)
    'dwa_weighted_salary': ['sum', 'mean'],
    
    # Most common seniority (mode)
    'seniority': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    
    # Calculate % female
    'sex_predicted': lambda x: (x == 'F').sum() / len(x) * 100,
    
    # Calculate % white
    'ethnicity_predicted': lambda x: (x == 'White').sum() / len(x) * 100
}).reset_index()

# Flatten column names and rename for clarity
df_dwa_clustered.columns = ['_'.join(str(c) for c in col).strip('_') if isinstance(col, tuple) else str(col) for col in df_dwa_clustered.columns.values]

# Debug: print columns after flattening
print(f"\nColumns after flattening: {df_dwa_clustered.columns.tolist()}")

# Create rename dictionary that handles various column name possibilities
rename_dict = {
    'DWA ID': 'dwa_id',
    'DWA_ID': 'dwa_id',
    'DWA Title': 'dwa_description',
    'DWA_Title': 'dwa_description',
    'DWA Title_first': 'dwa_description',
    'exposure': 'exposure',
    'exposure_first': 'exposure',
    'user_id_nunique': 'n_employees',
    'dwa_weighted_salary_sum': 'total_wagebill',
    'dwa_weighted_salary_mean': 'avg_wagebill_per_dwa',
    'seniority_<lambda>': 'most_common_seniority',
    'seniority': 'most_common_seniority',
    'sex_predicted_<lambda>': 'pct_female',
    'sex_predicted': 'pct_female',
    'ethnicity_predicted_<lambda>': 'pct_white',
    'ethnicity_predicted': 'pct_white'
}

df_dwa_clustered = df_dwa_clustered.rename(columns=rename_dict)

print(f"\n✓ Aggregation complete!")
print(f"  Output records: {len(df_dwa_clustered):,}")
print(f"  Columns: {df_dwa_clustered.columns.tolist()}")

print(f"\n📊 Aggregated Data Summary:")
print(f"  Total DWAs: {len(df_dwa_clustered):,}")
print(f"  Total wagebill across all DWAs: ${df_dwa_clustered['total_wagebill'].sum():,.2f}")
print(f"  Average employees per DWA: {df_dwa_clustered['n_employees'].mean():.2f}")
print(f"  Average total wagebill per DWA: ${df_dwa_clustered['total_wagebill'].mean():,.2f}")
print(f"  Average wagebill per employee-DWA: ${df_dwa_clustered['avg_wagebill_per_dwa'].mean():,.2f}")

print(f"  Average % female: {df_dwa_clustered['pct_female'].mean():.2f}%")
print(f"  Average % white: {df_dwa_clustered['pct_white'].mean():.2f}%")

# Sanity check: compare to total employee salaries
total_employee_salary = df_employees['salary'].sum()
total_dwa_wagebill = df_dwa_clustered['total_wagebill'].sum()
print(f"\n📊 Wagebill Sanity Check:")
print(f"  Total employee salaries (original):    ${total_employee_salary:,.2f}")
print(f"  Total DWA wagebill (weighted sum):     ${total_dwa_wagebill:,.2f}")
print(f"  Ratio (should be ~1.0):                 {total_dwa_wagebill / total_employee_salary:.3f}")
print(f"  Note: DWA wagebill should equal total employee salaries since each")


SECTION 2: CREATING DWA-LEVEL AGGREGATED DATA

Starting aggregation by DWA...
  Input records: 5,701,836
  Unique DWAs: 2,054

Columns after flattening: ['DWA ID', 'DWA Title_first', 'exposure_first', 'user_id_nunique', 'dwa_weighted_salary_sum', 'dwa_weighted_salary_mean', 'seniority_<lambda>', 'sex_predicted_<lambda>', 'ethnicity_predicted_<lambda>']

✓ Aggregation complete!
  Output records: 2,054
  Columns: ['dwa_id', 'dwa_description', 'exposure', 'n_employees', 'total_wagebill', 'avg_wagebill_per_dwa', 'most_common_seniority', 'pct_female', 'pct_white']

📊 Aggregated Data Summary:
  Total DWAs: 2,054
  Total wagebill across all DWAs: $27,800,086,970.90
  Average employees per DWA: 2775.97
  Average total wagebill per DWA: $13,534,609.04
  Average wagebill per employee-DWA: $4,873.11
  Average % female: 34.56%
  Average % white: 58.67%

📊 Wagebill Sanity Check:
  Total employee salaries (original):    $32,074,950,655.90
  Total DWA wagebill (weighted sum):     $27,800,086,970.90


In [45]:
# Save DWA-clustered data
output_file_dwa = f'{output_dir}/{company_code}_dwa_clustered.parquet'

# Delete existing file if it exists to ensure clean overwrite
if os.path.exists(output_file_dwa):
    os.remove(output_file_dwa)
    print(f"\n🗑️  Removed existing file: {output_file_dwa}")

print(f"\n💾 Saving DWA-clustered results to: {output_file_dwa}")
df_dwa_clustered.to_parquet(output_file_dwa, index=False, compression='snappy')

# Verify file was created and get size
file_size_mb_dwa = os.path.getsize(output_file_dwa) / (1024 * 1024)
print(f"✓ File saved successfully!")
print(f"  File size: {file_size_mb_dwa:.2f} MB")
print(f"  Records: {len(df_dwa_clustered):,}")

print(f"\n" + "="*70)
print("SECTION 2 COMPLETED SUCCESSFULLY")



🗑️  Removed existing file: output/Tables/AAPL_dwa_clustered.parquet

💾 Saving DWA-clustered results to: output/Tables/AAPL_dwa_clustered.parquet
✓ File saved successfully!
  File size: 0.16 MB
  Records: 2,054

SECTION 2 COMPLETED SUCCESSFULLY


In [46]:
# Display sample of DWA-clustered data
print("\n=== Sample DWA-Clustered Data ===")
display(df_dwa_clustered.head(10))

# Show some interesting statistics
print(f"\n📊 Top 5 DWAs by Total Wagebill:")
top_total_wagebill = df_dwa_clustered.nlargest(5, 'total_wagebill')[['dwa_id', 'dwa_description', 'n_employees', 'total_wagebill', 'avg_wagebill_per_dwa']]
for idx, row in top_total_wagebill.iterrows():
    print(f"  {row['dwa_id']} - {row['dwa_description'][:60]}...")
    print(f"    Total: ${row['total_wagebill']:,.2f} | Employees: {row['n_employees']} | Avg: ${row['avg_wagebill_per_dwa']:,.2f}")

print(f"\n📊 Top 5 DWAs by Number of Employees:")
top_employees = df_dwa_clustered.nlargest(5, 'n_employees')[['dwa_id', 'dwa_description', 'n_employees', 'total_wagebill']]
for idx, row in top_employees.iterrows():
    print(f"  {row['dwa_id']} - {row['dwa_description'][:60]}...")
    print(f"    Employees: {row['n_employees']} | Total wagebill: ${row['total_wagebill']:,.2f}")

print(f"\n📊 Data Quality Checks:")
print(f"  Records with exposure: {(~df_dwa_clustered['exposure'].isnull()).sum():,}")


=== Sample DWA-Clustered Data ===


,dwa_id,dwa_description,exposure,n_employees,total_wagebill,avg_wagebill_per_dwa,most_common_seniority,pct_female,pct_white
0,4.A.1.a.1.I01.D01,Review art or design materials.,0.762360,6115,87539427.127801,14322.55025,2,33.998365,68.094849
1,4.A.1.a.1.I01.D02,Study details of musical compositions.,0.658696,476,1289178.411844,2708.358008,2,22.478992,68.277311
2,4.A.1.a.1.I01.D03,Review production information to determine cos...,0.783656,59,436232.120773,7393.764759,2,62.711864,71.186441
3,4.A.1.a.1.I01.D04,Study scripts to determine project requirements.,0.634352,3474,13835680.078519,3982.636753,2,43.436960,65.659182
4,4.A.1.a.1.I01.D05,Review audio or video recordings.,NaN,101,0.0,<NA>,2,27.722772,58.415842
5,4.A.1.a.1.I02.D01,Read materials to determine needed actions.,0.857701,2645,5204567.500233,1967.700378,1,56.181474,56.862004
6,4.A.1.a.1.I02.D02,Read maps to determine routes.,0.753857,21,60513.066528,2881.574597,1,4.761905,33.333333
7,4.A.1.a.1.I02.D03,Review customer information.,0.602195,38,161490.00837,4249.737062,1,34.210526,50.000000
8,4.A.1.a.1.I02.D04,Read work orders or other instructions to dete...,0.543637,292,725882.563293,2485.899189,1,29.109589,57.876712
9,4.A.1.a.1.I02.D05,Read technical information needed to perform m...,0.459994,3452,4748687.634172,1376.032348,2,25.086906,60.892236



📊 Top 5 DWAs by Total Wagebill:
  4.A.4.a.2.I09.D09 - Communicate project information to others....
    Total: $885,908,651.85 | Employees: 36861 | Avg: $24,041.59
  4.A.2.b.2.I02.D07 - Modify software programs to improve performance....
    Total: $617,778,061.60 | Employees: 63035 | Avg: $9,804.29
  4.A.1.a.2.I10.D03 - Monitor computer system performance to ensure proper operati...
    Total: $616,658,089.67 | Employees: 67765 | Avg: $9,103.31
  4.A.4.b.6.I04.D02 - Provide technical support for software maintenance or use....
    Total: $596,463,831.02 | Employees: 69590 | Avg: $8,574.19
  4.A.4.a.2.I09.D07 - Collaborate with others to determine design specifications o...
    Total: $565,201,727.11 | Employees: 78169 | Avg: $7,485.22

📊 Top 5 DWAs by Number of Employees:
  4.A.4.a.2.I04.D03 - Collaborate with others to resolve information technology is...
    Employees: 78444 | Total wagebill: $400,321,300.84
  4.A.4.a.2.I09.D07 - Collaborate with others to determine design specific